In [ ]:
# Chunk 1 - Imports and path setup

import time
from pathlib import Path

import pandas as pd
import torch
import yaml
from ultralytics import YOLO
import matplotlib.pyplot as plt


def find_repo_root(start: Path = None, marker: str = ".git") -> Path:
    """Walk upward from `start` until a directory containing `marker` is found."""
    start = start or Path.cwd()
    for directory in [start, *start.parents]:
        if (directory / marker).exists():
            return directory
    raise FileNotFoundError(f"Could not find repo root (looked for '{marker}').")


REPO_ROOT = find_repo_root()
KFOLD_DIR = REPO_ROOT / "data" / "kfold"          # output of raw_to_yolo.ipynb, Chunk 9-10
TRAINING_OUTPUT_DIR = REPO_ROOT / "data" / "training_runs"
N_FOLDS = 5


def print_gpu_info() -> None:
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Device: {torch.cuda.get_device_name(0)}")


print_gpu_info()

In [ ]:
# Chunk 2 - Verify the folds are complete before training

def check_folds(kfold_dir: Path, n_folds: int) -> bool:
    """
    Verify each fold has a readable dataset.yaml and matching image/label
    counts in both train and val. Doesn't fix anything -- if this fails,
    re-run raw_to_yolo.ipynb's export rather than patching files by hand.
    """
    print("=" * 100)
    print(f"Checking {n_folds} folds in {kfold_dir}")
    all_ok = True

    for fold_idx in range(1, n_folds + 1):
        fold_dir = kfold_dir / f"fold{fold_idx}"
        yaml_path = fold_dir / "dataset.yaml"
        print(f"\nFold {fold_idx}: {fold_dir}")

        if not yaml_path.exists():
            print(f"  MISSING dataset.yaml")
            all_ok = False
            continue

        with open(yaml_path) as f:
            config = yaml.safe_load(f)
        print(f"  classes: {config.get('names')}")

        for split in ["train", "val"]:
            images = list((fold_dir / "images" / split).glob("*.jpg"))
            labels = list((fold_dir / "labels" / split).glob("*.txt"))
            status = "OK" if len(images) == len(labels) and len(images) > 0 else "MISMATCH"
            print(f"  {split}: {len(images)} images, {len(labels)} labels [{status}]")
            if status == "MISMATCH":
                all_ok = False

    summary_path = kfold_dir / "fold_summary.csv"
    if summary_path.exists():
        print("\nFold summary (from raw_to_yolo.ipynb):")
        print(pd.read_csv(summary_path).to_string(index=False))
    else:
        print("\nWARNING: fold_summary.csv not found -- re-run the export in raw_to_yolo.ipynb.")
        all_ok = False

    print("=" * 100)
    print("All folds OK." if all_ok else "Some folds have problems -- see above before training.")
    return all_ok


folds_ready = check_folds(KFOLD_DIR, N_FOLDS)

In [ ]:
# Chunk 3 - Training hyperparameters
#
# Final parameters from the masters thesis training runs. TEST_EPOCHS is for
# the quick sanity-check run in Chunk 4; EPOCHS is for the real run in Chunk 5.
# Everything else is shared between both runs via build_train_kwargs().

TRAINING_CONFIG = {
    "model_size": "yolov8s.pt", # you are free to try other sizes and iteratins of the yolo model
    "imgsz": 960, # make crabs more visible
    "batch_size": 16, # increase/decrase based on GPU/server
    "test_epochs": 5,
    "epochs": 100, # test as needed, based on thesis afgter epoch 100 the results converged

    "device": 0,
    "workers": 8,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "warmup_epochs": 3,
    "amp": True,
    "cache": "disk",
    "rect": False,
    "plots": True,
    "single_cls": True,   # only one class (crab) -- skips unnecessary per-class metric overhead
    "cos_lr": True,       # cosine learning rate schedule
    "crop_fraction": 0.8,

    "augmentation": {
        "mosaic": 0.5,
        "close_mosaic": 9,   # turn mosaic off for the last 9 epochs
        "mixup": 0.0,
        "degrees": 5.0,
        "translate": 0.15,
        "scale": 0.25,
        "fliplr": 0.5,
        "flipud": 0.0,       # no upside-down crabs
    },
}

config_path = TRAINING_OUTPUT_DIR / "training_config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, "w") as f:
    yaml.safe_dump(TRAINING_CONFIG, f, default_flow_style=False)

print("Training configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nSaved to: {config_path}")


def build_train_kwargs(data_yaml: Path, epochs: int, project: Path, name: str) -> dict:
    """
    Assemble model.train() kwargs from TRAINING_CONFIG. Shared between the
    test run (Chunk 4) and full run (Chunk 5) so both use identical settings
    except epoch count and output location.
    """
    aug = TRAINING_CONFIG["augmentation"]
    return dict(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=TRAINING_CONFIG["imgsz"],
        batch=TRAINING_CONFIG["batch_size"],
        device=TRAINING_CONFIG["device"],
        workers=TRAINING_CONFIG["workers"],
        optimizer=TRAINING_CONFIG["optimizer"],
        lr0=TRAINING_CONFIG["lr0"],
        lrf=TRAINING_CONFIG["lrf"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
        warmup_epochs=TRAINING_CONFIG["warmup_epochs"],
        amp=TRAINING_CONFIG["amp"],
        cache=TRAINING_CONFIG["cache"],
        rect=TRAINING_CONFIG["rect"],
        plots=TRAINING_CONFIG["plots"],
        single_cls=TRAINING_CONFIG["single_cls"],
        cos_lr=TRAINING_CONFIG["cos_lr"],
        crop_fraction=TRAINING_CONFIG["crop_fraction"],
        mosaic=aug["mosaic"],
        close_mosaic=aug["close_mosaic"],
        mixup=aug["mixup"],
        degrees=aug["degrees"],
        translate=aug["translate"],
        scale=aug["scale"],
        fliplr=aug["fliplr"],
        flipud=aug["flipud"],
        save=True,
        val=True,
        project=str(project),
        name=name,
    )

In [ ]:
# Chunk 4 - Test training run (sanity check, not real training)

def test_train(fold_idx: int = 1) -> None:
    data_yaml = KFOLD_DIR / f"fold{fold_idx}" / "dataset.yaml"
    print(f"Running a {TRAINING_CONFIG['test_epochs']}-epoch test on fold {fold_idx}...")

    kwargs = build_train_kwargs(
        data_yaml,
        epochs=TRAINING_CONFIG["test_epochs"],
        project=TRAINING_OUTPUT_DIR / "test_runs",
        name=f"fold{fold_idx}_test",
    )
    model = YOLO(TRAINING_CONFIG["model_size"])
    model.train(**kwargs)
    print("Test run complete -- check the plots/metrics above before running the full training in Chunk 5.")


test_train(fold_idx=1)

In [ ]:
# Chunk 5 - Full training across all folds

def train_fold(fold_idx: int) -> object:
    data_yaml = KFOLD_DIR / f"fold{fold_idx}" / "dataset.yaml"
    run_name = f"fold{fold_idx}_{TRAINING_CONFIG['model_size'].replace('.pt', '')}_{TRAINING_CONFIG['epochs']}epochs"

    print(f"\n{'=' * 80}\nFold {fold_idx} -- starting\n{'=' * 80}")

    kwargs = build_train_kwargs(
        data_yaml,
        epochs=TRAINING_CONFIG["epochs"],
        project=TRAINING_OUTPUT_DIR / "models",
        name=run_name,
    )
    kwargs["save_period"] = 25  # checkpoint every 25 epochs -- full runs only

    try:
        model = YOLO(TRAINING_CONFIG["model_size"])
        results = model.train(**kwargs)
        print(f"Fold {fold_idx} complete.")
        return results
    except Exception as e:
        print(f"Fold {fold_idx} FAILED: {e}")
        return None


if not folds_ready:
    raise RuntimeError("Folds failed the check in Chunk 2 -- fix that before training.")

all_results = []
start_time = time.time()

for fold_idx in range(1, N_FOLDS + 1):
    all_results.append(train_fold(fold_idx))
    if fold_idx < N_FOLDS:
        time.sleep(30)

hours = (time.time() - start_time) / 3600
print(f"\nAll {N_FOLDS} folds complete. Total time: {hours:.2f} hours.")
print(f"Results saved to: {TRAINING_OUTPUT_DIR / 'models'}")

In [ ]:
# Chunk 6 - Load training results from all folds

def fold_run_dir(fold_idx: int) -> Path:
    """Same run-name convention used in Chunk 5's train_fold()."""
    run_name = f"fold{fold_idx}_{TRAINING_CONFIG['model_size'].replace('.pt', '')}_{TRAINING_CONFIG['epochs']}epochs"
    return TRAINING_OUTPUT_DIR / "models" / run_name


def load_fold_results(fold_idx: int):
    """Load one fold's results.csv (per-epoch training/validation metrics)."""
    results_path = fold_run_dir(fold_idx) / "results.csv"
    if not results_path.exists():
        print(f"Fold {fold_idx}: no results.csv found at {results_path}")
        return None
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    return df


all_folds = []  # list of (fold_idx, df) -- kept paired so a missing fold can't misalign the rest
for fold_idx in range(1, N_FOLDS + 1):
    df = load_fold_results(fold_idx)
    if df is not None:
        all_folds.append((fold_idx, df))

if not all_folds:
    raise RuntimeError("No fold results found -- check TRAINING_OUTPUT_DIR and that training finished.")

print(f"Loaded results for {len(all_folds)}/{N_FOLDS} folds.")

In [ ]:
# Chunk 7 - Plot key metrics across all folds

fold_colors = plt.cm.tab10.colors
metric_plots = [
    ("metrics/mAP50(B)", "mAP50"),
    ("metrics/mAP50-95(B)", "mAP50-95"),
    ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)", "Recall"),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("K-fold Training Results", fontsize=16)

for ax, (column, title) in zip(axes.flat, metric_plots):
    for fold_idx, df in all_folds:
        ax.plot(df["epoch"], df[column], color=fold_colors[(fold_idx - 1) % len(fold_colors)],
                alpha=0.85, label=f"Fold {fold_idx}")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = TRAINING_OUTPUT_DIR / "kfold_training_plots.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"Plot saved to: {plot_path}")
plt.show()

In [ ]:
# Chunk 8 - Per-fold summary metrics, with a fresh validation pass

def summarize_fold(fold_idx: int, df: pd.DataFrame) -> dict:
    """
    Pull final/best metrics from results.csv, then re-run model.val() on that
    fold's best weights against its own validation set -- a clean, explicit
    check rather than only trusting the last row of the training log.
    """
    weights_path = fold_run_dir(fold_idx) / "weights" / "best.pt"
    data_yaml = KFOLD_DIR / f"fold{fold_idx}" / "dataset.yaml"

    summary = {
        "fold": fold_idx,
        "best_mAP50": df["metrics/mAP50(B)"].max(),
        "final_mAP50": df["metrics/mAP50(B)"].iloc[-1],
        "final_mAP50-95": df["metrics/mAP50-95(B)"].iloc[-1],
        "final_precision": df["metrics/precision(B)"].iloc[-1],
        "final_recall": df["metrics/recall(B)"].iloc[-1],
    }

    if weights_path.exists():
        model = YOLO(str(weights_path))
        metrics = model.val(data=str(data_yaml), conf=0.25, iou=0.7, save_conf=True, save_txt=True)
        summary.update({
            "val_mAP50": metrics.box.map50,
            "val_mAP50-95": metrics.box.map,
            "val_precision": metrics.box.mp,
            "val_recall": metrics.box.mr,
        })
    else:
        print(f"Fold {fold_idx}: no best.pt found at {weights_path}, skipping re-validation.")

    return summary


summary_df = pd.DataFrame(summarize_fold(fold_idx, df) for fold_idx, df in all_folds)

print("\nPer-fold results:")
print(summary_df.to_string(index=False))

print("\nAcross all folds (mean +/- std):")
for col in [c for c in summary_df.columns if c != "fold"]:
    print(f"  {col}: {summary_df[col].mean():.4f} +/- {summary_df[col].std():.4f}")

summary_csv_path = TRAINING_OUTPUT_DIR / "kfold_results_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nSaved to: {summary_csv_path}")